# Avito Services: Candidate Generation — Финальное решение

Один файл, все от загрузки данных до answer.csv.

# Архитектура (из эксп. журнала решения с Recall@50=0.88 на платформе)

## 1. BM25F + Гео (главный прыжок: 0.30 -> ~0.85)
- Раздельная нормировка полей: title w=20 b=0.6, desc w=1 b=0.75 (k1=1.2)
- Гео добавляется КО ВСЕМ ненулевым BM25F-позициям (не к топ-N!)
  Причина: "маникюр" -> 1000+ объявлений с близким BM25F-скором,
  гео сужает набор ближайшими. Обрезка до топ-N ДО гео теряет 13.6% релевантных.
- BM25F нормируется внутри запроса, geo_weight=0.10

## 2. Dense Retrieval (+0.013 к слиянию)
- Модель: multilingual-e5-small (117MB) / e5-base (471MB, лучше, нужен GPU)
- Fine-tuning на positive парах из train (если есть GPU), 2 эпохи
- In-batch negatives ТОЛЬКО — не hard negatives!
  Hard negatives вредны: одинаковый заголовок в другой локации — самый
  тяжелый лексический негатив, но текст их не различает, различает гео.
  Обучение на них дает противоречивый сигнал энкодеру.
- Приносит кандидатов с семантическим разрывом (маникюр vs наращивание ногтей)
- Dense weight=0.50 (vs geo weight=0.10): несет сопоставимый объем информации

## 3. Train Lookup (Recall=1.0 для 37% известных запросов)
- (q, cat), (q, loc), (q): Counter кликов из train
- Добавляется как сильный бустер поверх основного скора

## Итоговый скор
    score(q, d) = bm25f_norm + 0.10*geo + 0.50*dense + lookup_boost

## Установка зависимостей

In [1]:
!pip install -q rank-bm25 sentence-transformers faiss-cpu PyStemmer pyarrow tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 90.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 747.7/747.7 kB 53.2 MB/s eta 0:00:00


In [2]:
!pip install sentence-transformers faiss-cpu

## Загрузка данных с Яндекс.Диска

In [3]:
import os, requests
from pathlib import Path

PUBLIC_URL = "https://disk.yandex.ru/d/sNhfo0YOjGtufg"

print("Получаем прямую ссылку для скачивания через API Яндекс.Диска...")
api_url  = f"https://cloud-api.yandex.net/v1/disk/public/resources/download?public_key={PUBLIC_URL}"
response = requests.get(api_url)

if response.status_code == 200:
    download_url = response.json()["href"]
    print("Прямая ссылка получена!")
else:
    raise RuntimeError(f"Ошибка API Яндекс.Диска: {response.status_code}")

archive_name = "avito_data.zip"
print("Скачиваем архив...")
os.system(f'wget -q --show-progress -O {archive_name} "{download_url}"')

DATA_DIR = Path("/content/first")
DATA_DIR.mkdir(exist_ok=True)
os.system(f"unzip -q -o {archive_name} -d {DATA_DIR}")

archive_name2 = "/content/first/NLP_avito_interns/dataset.zip"
DATA_DIR = Path("/content/data")
DATA_DIR.mkdir(exist_ok=True)
os.system(f"unzip -q -o {archive_name2} -d {DATA_DIR}")

parquet_files = list(DATA_DIR.rglob("*.parquet"))
for p in parquet_files:
    target = DATA_DIR / p.name
    if not target.exists():
        p.rename(target)

print("Данные готовы:")
os.system(f"ls -lh {DATA_DIR}/*.parquet")

Получаем прямую ссылку для скачивания через API Яндекс.Диска...
Прямая ссылка получена!
Скачиваем архив...
Данные готовы:


0

## Импорты и конфигурация

In [4]:
import re, pickle, time, json
from math import radians, sin, cos, sqrt, atan2
from collections import defaultdict, Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

# BM25F
try:
    pass  # реализован ниже
except Exception:
    pass

# Стемминг (опционально)
try:
    from Stemmer import Stemmer as SnowballStemmer
    _stemmer = SnowballStemmer("russian")
    HAS_STEM = True
    print("[OK] PyStemmer — стемминг включен")
except ImportError:
    HAS_STEM = False
    print("[INFO] Без стемминга (pip install PyStemmer)")

# Dense retrieval
try:
    from sentence_transformers import SentenceTransformer, InputExample, losses
    from torch.utils.data import DataLoader
    import faiss
    HAS_DENSE = True
    print("[OK] sentence-transformers + faiss")
except ImportError:
    HAS_DENSE = False
    print("[INFO] Dense retrieval недоступен (pip install sentence-transformers faiss-cpu)")

SEED       = 42
DATA_DIR   = Path("/content/data")
SAVE_DIR   = Path("/content/artifacts")
SAVE_DIR.mkdir(exist_ok=True)
N_CANDS    = 50
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

# BM25F параметры (из эксп. журнала топ-решения: 84 конфигурации на сетке)
FIELD_W   = {"title": 20.0, "desc": 1.0}
FIELD_B   = {"title": 0.6,  "desc": 0.75}
BM25F_K1  = 1.2
IDF_FLOOR = 0.25

# Гео (внутренний максимум на сетке весов: 0.00->0.8837 при 0.10, спад при 0.20)
GEO_W     = 0.10
GEO_MAX   = 30.0    # км, при котором бонус падает до 0

# Dense
DENSE_MODEL  = "intfloat/multilingual-e5-small"   # поменяйте на e5-base для +качества
DENSE_EPOCHS = 2
DENSE_BATCH  = 64    # for fine-tuning (in-batch negatives = больше пар в батче лучше)
ENC_BATCH    = 256   # для кодирования корпуса
DENSE_TOP_K  = 1000  # кандидатов из FAISS (1000 vs 200 поднимает потолок на +0.007)
DENSE_W      = 0.50  # вес dense в итоговом скоре

# Lookup boost (поверх BM25F+geo+dense)
LU_CAT_W  = 3.0   # (q, category)
LU_LOC_W  = 2.0   # (q, location)
LU_GL_W   = 1.5   # (q only)

np.random.seed(SEED)
torch.manual_seed(SEED)

[OK] PyStemmer — стемминг включен
[OK] sentence-transformers + faiss
Device: cuda


/tmp/ipykernel_747/2370138848.py:29: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import SentenceTransformer, InputExample, losses


## Загрузка данных

In [5]:
print("Загружаем данные...")
train             = pd.read_parquet(DATA_DIR / "train.parquet")
benchmark_queries = pd.read_parquet(DATA_DIR / "benchmark_queries.parquet")
benchmark_items   = pd.read_parquet(DATA_DIR / "benchmark_items.parquet")

for df, col in [(benchmark_items, "item_id"),
                (benchmark_queries, "query_id"),
                (train, "item_id")]:
    df[col] = df[col].astype(str)

VALID_IDS   = set(benchmark_items["item_id"])
ALL_IDS     = benchmark_items["item_id"].tolist()
ID_TO_IDX   = {iid: i for i, iid in enumerate(ALL_IDS)}

print(f"train: {len(train):,} | items: {len(benchmark_items):,} | queries: {len(benchmark_queries):,}")

Загружаем данные...
train: 497,673 | items: 189,212 | queries: 2,452


## Предобработка текста

In [6]:
def tokenize(text: str) -> list:
    """Нижний регистр, только [а-яa-z0-9], длина >= 2, опционально стемминг."""
    if not text or (isinstance(text, float) and np.isnan(text)):
        return []
    text = str(text).lower()
    text = re.sub(r"[^а-яa-z0-9\s]", " ", text)
    tokens = [t for t in text.split() if len(t) >= 2]
    if HAS_STEM and tokens:
        tokens = _stemmer.stemWords(tokens)
    return tokens


def item_fields(row: pd.Series) -> dict:
    """
    Поля объявления для BM25F: title + description[:500].
    infm_params исключен: добавление снижает Recall с 0.28 до 0.26
    (родовые фасетные значения общие для тысяч объявлений, разбавляют вес нужных токенов).
    """
    out = {}
    t = row.get("item_title_raw", "")
    if t and not (isinstance(t, float) and np.isnan(t)):
        out["title"] = tokenize(str(t))
    d = row.get("item_description_raw", "")
    if d and not (isinstance(d, float) and np.isnan(d)):
        out["desc"] = tokenize(str(d)[:500])
    return out


def query_toks(row: pd.Series) -> list:
    """Только search_query. Текст фильтра не добавляем (эксп: не работает)."""
    q = row.get("search_query", "")
    if q and not (isinstance(q, float) and np.isnan(q)):
        return tokenize(str(q))
    return []


def item_text_for_dense(row: pd.Series) -> str:
    """Текст объявления для dense retrieval (без field-weighting, просто строка)."""
    parts = []
    t = row.get("item_title_raw", "")
    if t and not (isinstance(t, float) and np.isnan(t)): parts.append(str(t))
    d = row.get("item_description_raw", "")
    if d and not (isinstance(d, float) and np.isnan(d)): parts.append(str(d)[:300])
    return " ".join(parts)

## BM25F

In [7]:
class BM25F:
    """
    BM25F с раздельной нормировкой полей и обратным индексом.

    Формула:
        wtf(t,d) = sum_f w_f * tf(t,d_f) / (1 - b_f + b_f * |d_f| / avgdl_f)
        score(q,d) = sum_{t in q} IDF(t) * wtf(t,d) / (k1 + wtf(t,d))

    Обратный индекс: term -> (doc_ids: int32, wtfs: float32)
    Скоринг O(postings) вместо O(n_docs). Размер ~100MB vs 290MB у BM25Okapi.
    """

    def __init__(self, field_docs, field_weights, field_b, k1=1.2, idf_floor=0.25):
        self.k1 = k1
        self.n  = len(field_docs)
        fields  = list(field_weights.keys())

        avgdl = {f: sum(len(d.get(f, [])) for d in field_docs) / max(self.n, 1)
                 for f in fields}

        # Однопроходное построение
        term_wtf = defaultdict(lambda: defaultdict(float))
        term_df  = defaultdict(set)

        for doc_id, doc in enumerate(tqdm(field_docs, desc="  BM25F build",
                                          leave=False)):
            for f in fields:
                tokens = doc.get(f, [])
                if not tokens: continue
                denom = 1.0 - field_b[f] + field_b[f] * len(tokens) / avgdl[f]
                w     = field_weights[f]
                for term, tf in Counter(tokens).items():
                    term_wtf[term][doc_id] += w * tf / denom
                    term_df[term].add(doc_id)

        n = self.n
        self.idf = {t: max(np.log((n - len(d) + 0.5) / (len(d) + 0.5)), idf_floor)
                    for t, d in term_df.items()}

        self.inv = {}
        for term, dw in term_wtf.items():
            ids  = np.array(list(dw.keys()),    dtype=np.int32)
            wtfs = np.array(list(dw.values()),  dtype=np.float32)
            self.inv[term] = (ids, wtfs)

        avg_t = avgdl.get("title", 0); avg_d = avgdl.get("desc", 0)
        print(f"  BM25F: {self.n:,} docs, {len(self.inv):,} terms, "
              f"avgdl title={avg_t:.1f} desc={avg_d:.1f}")

    def get_scores_sparse(self, qtoks):
        """Ненулевые BM25F-скоры: (doc_ids, scores). Sparse, без аллокации n-вектора."""
        acc = {}
        k1  = self.k1
        for term in set(qtoks):
            if term not in self.inv: continue
            idf = self.idf.get(term, 0.0)
            if idf <= 0: continue
            ids, wtfs = self.inv[term]
            cs = idf * wtfs / (k1 + wtfs)
            for did, c in zip(ids.tolist(), cs.tolist()):
                acc[did] = acc.get(did, 0.0) + c
        if not acc:
            return np.empty(0, dtype=np.int32), np.empty(0, dtype=np.float32)
        ids    = np.array(list(acc.keys()),   dtype=np.int32)
        scores = np.array(list(acc.values()), dtype=np.float32)
        return ids, scores

## Гео: центроиды локаций + Haversine

In [8]:
def _hav(lat1, lon1, lat2, lon2):
    R = 6371.0
    dlat = radians(lat2-lat1); dlon = radians(lon2-lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1))*cos(radians(lat2))*sin(dlon/2)**2
    return 2*R*atan2(sqrt(a), sqrt(1-a))


print("Строим гео-кеши...")
_lats = pd.to_numeric(benchmark_items["item_latitude"],  errors="coerce")
_lons = pd.to_numeric(benchmark_items["item_longitude"], errors="coerce")
_ok   = ~_lats.isna() & ~_lons.isna()

item_geo = dict(zip(benchmark_items.loc[_ok,"item_id"],
                    zip(_lats[_ok].values, _lons[_ok].values)))
print(f"  item_geo: {len(item_geo):,} объявлений")

loc_cen = {}
if "item_location_id" in benchmark_items.columns:
    _tmp = benchmark_items.assign(_la=_lats, _lo=_lons).dropna(subset=["_la","_lo"])
    for loc, g in _tmp.groupby("item_location_id"):
        loc_cen[int(loc)] = (float(g["_la"].median()), float(g["_lo"].median()))
    print(f"  loc_centroid: {len(loc_cen):,} локаций")
del _lats, _lons, _ok, _tmp


def geo_bonus(q_loc: int, item_id: str) -> float:
    """Линейный гео-бонус [0,1]: 1.0 при 0 км, 0.0 при >= GEO_MAX км."""
    if q_loc not in loc_cen or item_id not in item_geo: return 0.0
    qlat, qlon = loc_cen[q_loc]
    ilat, ilon = item_geo[item_id]
    return max(0.0, 1.0 - _hav(qlat, qlon, ilat, ilon) / GEO_MAX)

Строим гео-кеши...
  item_geo: 189,211 объявлений
  loc_centroid: 2,877 локаций


## Train Lookup

In [9]:
print("Строим train lookup...")
t0 = time.time()

valid_train = train[train["item_id"].isin(VALID_IDS)].copy()
valid_train["_q"]   = valid_train["search_query"].fillna("").str.lower().str.strip()
valid_train["_cat"] = valid_train["search_category"].fillna("").astype(str) \
    if "search_category" in valid_train.columns else ""
valid_train["_loc"] = valid_train["search_location_id"].fillna(-1).astype(int).astype(str) \
    if "search_location_id" in valid_train.columns else "-1"

prod_cat = defaultdict(Counter)
prod_loc = defaultdict(Counter)
prod_gl  = defaultdict(Counter)
for _, r in tqdm(valid_train.iterrows(), total=len(valid_train),
                 desc="  lookup", leave=False):
    q = r["_q"]; iid = r["item_id"]
    prod_cat[(q, r["_cat"])][iid] += 1
    if r["_loc"] != "-1": prod_loc[(q, r["_loc"])][iid] += 1
    prod_gl[q][iid] += 1

print(f"  (q,cat): {len(prod_cat):,} | (q,loc): {len(prod_loc):,} | q: {len(prod_gl):,} | {time.time()-t0:.1f}с")

Строим train lookup...


  lookup:   0%|          | 0/33010 [00:00<?, ?it/s]

  (q,cat): 12,209 | (q,loc): 24,307 | q: 12,208 | 3.6с


## BM25F: токенизация и построение индекса

In [10]:
print("Токенизируем объявления...")
t0 = time.time()
field_docs = [item_fields(r) for _, r in
              tqdm(benchmark_items.iterrows(), total=len(benchmark_items),
                   desc="  tokenize", leave=False)]

print(f"  Ср. токенов: title={np.mean([len(d.get('title',[])) for d in field_docs]):.1f}, "
      f"desc={np.mean([len(d.get('desc',[])) for d in field_docs]):.1f}")

bm25f = BM25F(field_docs, FIELD_W, FIELD_B, k1=BM25F_K1, idf_floor=IDF_FLOOR)
print(f"  BM25F построен за {time.time()-t0:.1f}с")

with open(SAVE_DIR / "bm25f.pkl", "wb") as f: pickle.dump(bm25f, f)
print(f"  -> bm25f.pkl ({(SAVE_DIR/'bm25f.pkl').stat().st_size/1024/1024:.0f} MB)")

Токенизируем объявления...


  tokenize:   0%|          | 0/189212 [00:00<?, ?it/s]

  Ср. токенов: title=4.3, desc=50.7


  BM25F build:   0%|          | 0/189212 [00:00<?, ?it/s]

  BM25F: 189,212 docs, 196,287 terms, avgdl title=4.3 desc=50.7
  BM25F построен за 70.6с
  -> bm25f.pkl (89 MB)


## Dense Retrieval

In [11]:
dense_model       = None
item_emb_matrix   = None   # (n_items, emb_dim) float32
faiss_index       = None

if HAS_DENSE:
    # ---
    # Загрузка модели
    # ---
    print(f"\nЗагружаем {DENSE_MODEL}...")
    dense_model = SentenceTransformer(DENSE_MODEL, device=DEVICE)
    emb_dim     = dense_model.get_sentence_embedding_dimension()
    print(f"  Параметры: {sum(p.numel() for p in dense_model.parameters()):,}, dim={emb_dim}")

    # ---
    # Fine-tuning (если есть GPU)
    # ---
    if torch.cuda.is_available():
        print("\nFine-tuning на train pairs (in-batch negatives)...")
        print("  Почему in-batch, а не hard negatives:")
        print("  Одинаковый заголовок в другой локации = самый тяжелый лекс. негатив,")
        print("  но текст их не различает. Энкодер получает противоречивый сигнал.")
        print("  Разводит их гео — оно для этого и стоит в слиянии.")

        # Positive pairs: (query_text, item_text) из train
        # Используем train напрямую: item_title_raw + description есть в train
        pairs_src = train.drop_duplicates(subset=["search_query","item_id"])
        pairs_src = pairs_src[pairs_src["item_id"].isin(VALID_IDS)].copy()

        # Текст запроса
        q_texts = pairs_src["search_query"].fillna("").astype(str).tolist()

        # Текст объявления (из train — там уже есть поля)
        def _item_txt_from_train(row):
            parts = []
            t = row.get("item_title_raw", "")
            if t and not (isinstance(t, float) and np.isnan(t)):
                parts.append(str(t))
            d = row.get("item_description_raw", "")
            if d and not (isinstance(d, float) and np.isnan(d)):
                parts.append(str(d)[:300])
            return " ".join(parts)

        i_texts = [_item_txt_from_train(r) for _, r in
                   tqdm(pairs_src.iterrows(), total=len(pairs_src),
                        desc="  item texts", leave=False)]

        # Создаем InputExample для sentence-transformers
        # Инструкция "query:" / "passage:" обязательна для e5-моделей
        MAX_PAIRS = 200_000  # ограничиваем для скорости
        examples = [
            InputExample(texts=["query: " + q, "passage: " + i])
            for q, i in zip(q_texts[:MAX_PAIRS], i_texts[:MAX_PAIRS])
            if q.strip() and i.strip()
        ]
        print(f"  Training pairs: {len(examples):,}")

        loader    = DataLoader(examples, shuffle=True, batch_size=DENSE_BATCH, drop_last=True)
        loss_fn   = losses.MultipleNegativesRankingLoss(dense_model)
        warmup    = int(len(loader) * DENSE_EPOCHS * 0.05)

        print(f"  Батчей: {len(loader)}, эпохи: {DENSE_EPOCHS}, warmup: {warmup}")
        dense_model.fit(
            train_objectives=[(loader, loss_fn)],
            epochs=DENSE_EPOCHS,
            warmup_steps=warmup,
            show_progress_bar=True,
            output_path=str(SAVE_DIR / "dense_model_ft"),
        )
        dense_model = SentenceTransformer(str(SAVE_DIR / "dense_model_ft"), device=DEVICE)
        print("  Fine-tuning завершен, модель сохранена")
    else:
        print("\n  GPU не найден — используем pre-trained модель без fine-tuning")
        print("  Для fine-tuning нужен GPU (Colab: Runtime -> Change runtime type -> T4)")

    # ---
    # Кодирование корпуса
    # ---
    print(f"\nКодируем {len(benchmark_items):,} объявлений...")
    t0 = time.time()
    item_texts_dense = ["passage: " + item_text_for_dense(r)
                        for _, r in benchmark_items.iterrows()]

    item_emb_matrix = dense_model.encode(
        item_texts_dense,
        batch_size=ENC_BATCH,
        normalize_embeddings=True,
        show_progress_bar=True,
        convert_to_numpy=True,
        device=DEVICE,
    ).astype("float32")
    print(f"  Embeddings: {item_emb_matrix.shape}, {time.time()-t0:.1f}с")

    # FAISS IndexFlatIP (точный поиск, cosine при нормализованных векторах)
    faiss_index = faiss.IndexFlatIP(emb_dim)
    faiss_index.add(item_emb_matrix)
    print(f"  FAISS готов: {faiss_index.ntotal:,} векторов")

    np.save(SAVE_DIR / "item_emb.npy", item_emb_matrix)
    faiss.write_index(faiss_index, str(SAVE_DIR / "faiss.bin"))
    print("  -> item_emb.npy + faiss.bin")
else:
    print("\n[WARN] Dense retrieval недоступен — используем только BM25F + гео")


Загружаем intfloat/multilingual-e5-small...


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/498k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  Fine-tuning завершен, модель сохранена

Кодируем 189,212 объявлений...


Batches:   0%|          | 0/740 [00:00<?, ?it/s]

  Embeddings: (189212, 384), 629.5с
  FAISS готов: 189,212 векторов
  -> item_emb.npy + faiss.bin


## OOD Validation Split

500 запросов исключены из lookup для честной оценки BM25F.

In [12]:
unique_q = valid_train["_q"].unique().copy()
np.random.seed(SEED); np.random.shuffle(unique_q)

val_q_set = set(unique_q[:500])

val_ood = (
    valid_train[valid_train["_q"].isin(val_q_set)]
    .groupby("search_query")["item_id"].apply(set).reset_index()
)
val_ood.columns = ["search_query", "relevant_ids"]

# Val-lookup: без OOD-запросов
val_gl  = defaultdict(Counter)
val_cat = defaultdict(Counter)
val_loc = defaultdict(Counter)
for _, r in valid_train[~valid_train["_q"].isin(val_q_set)].iterrows():
    q = r["_q"]; iid = r["item_id"]
    val_cat[(q, r["_cat"])][iid] += 1
    if r["_loc"] != "-1": val_loc[(q, r["_loc"])][iid] += 1
    val_gl[q][iid] += 1

print(f"OOD validation: {len(val_ood)} запросов")

OOD validation: 500 запросов


## Основная функция скоринга

In [13]:
def _get_loc(q_row):
    v = q_row.get("search_location_id", None)
    if v is None or (isinstance(v, float) and np.isnan(v)): return -1
    return int(v)


def score_candidates(q_row: pd.Series,
                     bm25f_obj, faiss_idx, item_embs,
                     lu_gl, lu_cat, lu_loc,
                     n: int = N_CANDS,
                     dense_model_obj=None) -> list:
    """
    Скоринг кандидатов для одного запроса.

    Порядок:
    1. BM25F: все ненулевые позиции -> нормировать -> добавить гео * GEO_W
    2. Dense: top-DENSE_TOP_K -> добавить dense_score * DENSE_W
    3. Lookup: (q,cat), (q,loc), (q) -> добавить lookup_boost
    4. Сортировать, вернуть топ-n
    """
    q_text = str(q_row.get("search_query", "")).lower().strip()
    q_cat  = str(q_row.get("search_category", ""))
    q_loc  = _get_loc(q_row)

    scores = {}  # item_id -> float

    # --- BM25F + гео ---
    qtoks = query_toks(q_row)
    if qtoks:
        doc_ids, raw_s = bm25f_obj.get_scores_sparse(qtoks)
        if len(doc_ids) > 0:
            max_s = float(raw_s.max()); max_s = max_s if max_s > 0 else 1.0
            norm_s = raw_s / max_s
            for did, ns in zip(doc_ids.tolist(), norm_s.tolist()):
                iid = ALL_IDS[did]
                g   = geo_bonus(q_loc, iid) if q_loc != -1 else 0.0
                scores[iid] = ns + GEO_W * g

    # --- Dense (если доступно) ---
    if faiss_idx is not None and dense_model_obj is not None and item_embs is not None:
        q_full  = "query: " + str(q_row.get("search_query", ""))
        q_enc   = dense_model_obj.encode(
            [q_full], normalize_embeddings=True,
            show_progress_bar=False, convert_to_numpy=True,
        ).astype("float32")
        d_scores, d_idx = faiss_idx.search(q_enc, DENSE_TOP_K)
        max_ds  = float(d_scores[0][0]) if d_scores[0][0] > 0 else 1.0
        for ds, di in zip(d_scores[0], d_idx[0]):
            if di < 0: continue
            iid = ALL_IDS[di]
            scores[iid] = scores.get(iid, 0.0) + DENSE_W * float(ds) / max_ds

    # --- Train Lookup (бустер) ---
    def _lu(ctr, w):
        if not ctr: return
        mx = max(ctr.values())
        for iid, cnt in ctr.items():
            if iid in VALID_IDS:
                scores[iid] = scores.get(iid, 0.0) + w * cnt / mx

    _lu(lu_cat.get((q_text, q_cat), {}), LU_CAT_W)
    _lu(lu_loc.get((q_text, str(q_loc)), {}), LU_LOC_W)
    _lu(lu_gl.get(q_text, {}),               LU_GL_W)

    return [iid for iid, _ in sorted(scores.items(), key=lambda x: -x[1])
            if iid in VALID_IDS][:n]


def get_candidates(q_row: pd.Series, n: int = N_CANDS) -> list:
    """Production wrapper, использует глобальные объекты."""
    return score_candidates(
        q_row, bm25f, faiss_index, item_emb_matrix,
        prod_gl, prod_cat, prod_loc, n=n,
        dense_model_obj=dense_model,
    )

## Offline Validation

In [14]:
print("\nOffline validation на OOD-запросах...")
print("(lookup возвращает 0 — запросы намеренно вне lookup)")

recalls = []
for _, row in tqdm(val_ood.iterrows(), total=len(val_ood),
                   desc="  validation", leave=False):
    rel   = {str(i) for i in row["relevant_ids"]} & VALID_IDS
    if not rel: continue
    cands = score_candidates(
        pd.Series({"search_query": row["search_query"]}),
        bm25f, faiss_index, item_emb_matrix,
        val_gl, val_cat, val_loc,
        dense_model_obj=dense_model,
    )
    recalls.append(len(rel & set(cands)) / len(rel))

val_recall = float(np.mean(recalls)) if recalls else 0.0
print(f"\n  Recall@{N_CANDS} [OOD]: {val_recall:.4f}  ({len(recalls)} запросов)")
print()
print("  Компоненты:")
print(f"    BM25F:  field-norm, title w=20 b=0.6, desc w=1 b=0.75")
print(f"    Гео:    Haversine centroid, weight={GEO_W}")
print(f"    Dense:  {'fine-tuned' if torch.cuda.is_available() and HAS_DENSE else 'pre-trained или отсутствует'}, weight={DENSE_W}")
print()
print("  Ожидания из эксп. журнала топ-решения:")
print("    BM25F alone:   ~0.30")
print("    + гео (norm):  ~0.50-0.65  (у топ-решения: 0.88 на known-запросах)")
print("    + dense:       ~0.60-0.75")


Offline validation на OOD-запросах...
(lookup возвращает 0 — запросы намеренно вне lookup)


  validation:   0%|          | 0/500 [00:00<?, ?it/s]


  Recall@50 [OOD]: 0.4040  (500 запросов)

  Компоненты:
    BM25F:  field-norm, title w=20 b=0.6, desc w=1 b=0.75
    Гео:    Haversine centroid, weight=0.1
    Dense:  fine-tuned, weight=0.5

  Ожидания из эксп. журнала топ-решения:
    BM25F alone:   ~0.30
    + гео (norm):  ~0.50-0.65  (у топ-решения: 0.88 на known-запросах)
    + dense:       ~0.60-0.75


## Генерация кандидатов для всех benchmark-запросов

In [15]:
print(f"\nГенерируем кандидатов для {len(benchmark_queries):,} запросов...")

# Если dense доступен и нужно кодировать все запросы батчем (быстрее)
if HAS_DENSE and dense_model is not None and faiss_index is not None:
    print("  Pre-encoding запросов батчем...")
    all_q_texts  = ["query: " + str(r.get("search_query",""))
                    for _, r in benchmark_queries.iterrows()]
    all_q_embs   = dense_model.encode(
        all_q_texts, batch_size=ENC_BATCH,
        normalize_embeddings=True, show_progress_bar=True,
        convert_to_numpy=True,
    ).astype("float32")
    print("  Batch FAISS search...")
    d_scores_all, d_idx_all = faiss_index.search(all_q_embs, DENSE_TOP_K)
    print("  Готово")
else:
    all_q_embs = None
    d_scores_all = d_idx_all = None

all_qids  = []
all_preds = []

for i, (_, q_row) in enumerate(tqdm(benchmark_queries.iterrows(),
                                     total=len(benchmark_queries), desc="queries")):
    qid    = str(q_row["query_id"])
    q_text = str(q_row.get("search_query", "")).lower().strip()
    q_cat  = str(q_row.get("search_category", ""))
    q_loc  = _get_loc(q_row)

    scores = {}

    # BM25F + гео
    qtoks = query_toks(q_row)
    if qtoks:
        doc_ids, raw_s = bm25f.get_scores_sparse(qtoks)
        if len(doc_ids) > 0:
            max_s = float(raw_s.max()); max_s = max_s if max_s > 0 else 1.0
            ns_arr = raw_s / max_s
            for did, ns in zip(doc_ids.tolist(), ns_arr.tolist()):
                iid = ALL_IDS[did]
                g   = geo_bonus(q_loc, iid) if q_loc != -1 else 0.0
                scores[iid] = ns + GEO_W * g

    # Dense (batch mode)
    if d_idx_all is not None:
        ds_row  = d_scores_all[i]
        di_row  = d_idx_all[i]
        max_ds  = float(ds_row[0]) if ds_row[0] > 0 else 1.0
        for ds, di in zip(ds_row, di_row):
            if di < 0: continue
            iid = ALL_IDS[di]
            scores[iid] = scores.get(iid, 0.0) + DENSE_W * float(ds) / max_ds

    # Lookup
    for ctr, w in [
        (prod_cat.get((q_text, q_cat), {}), LU_CAT_W),
        (prod_loc.get((q_text, str(q_loc)), {}), LU_LOC_W),
        (prod_gl.get(q_text, {}),               LU_GL_W),
    ]:
        if not ctr: continue
        mx = max(ctr.values())
        for iid, cnt in ctr.items():
            if iid in VALID_IDS:
                scores[iid] = scores.get(iid, 0.0) + w * cnt / mx

    top50 = [iid for iid, _ in sorted(scores.items(), key=lambda x: -x[1])
             if iid in VALID_IDS][:N_CANDS]

    all_qids.append(qid)
    all_preds.append(top50)

cand_counts = [len(p) for p in all_preds]
print(f"\nКандидатов: min={min(cand_counts)}, mean={np.mean(cand_counts):.1f}, max={max(cand_counts)}")
print(f"Запросов с < {N_CANDS} кандидатами: {sum(c < N_CANDS for c in cand_counts)}")


Генерируем кандидатов для 2,452 запросов...
  Pre-encoding запросов батчем...


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

  Batch FAISS search...
  Готово


queries:   0%|          | 0/2452 [00:00<?, ?it/s]


Кандидатов: min=50, mean=50.0, max=50
Запросов с < 50 кандидатами: 0


## Сохранение answer.csv

In [16]:
answer = pd.DataFrame({
    "query_id": all_qids,
    "answer":   [" ".join(p) for p in all_preds],
})

OUT = "answer.csv"
answer.to_csv(OUT, index=False, encoding="utf-8")
print(f"Сохранено: {OUT} ({len(answer)} строк)")
print(answer.head(3).to_string(index=False))

Сохранено: answer.csv (2452 строк)
        query_id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            answer
70DfDUpwjxB4lzFd d01e14d2ffd4ddba 66860b3fd69e8056 d722bcda1a555091 2d73dacac243ce62 9515d1e1ecdac

## Проверка формата

In [17]:
print("\nПроверяем формат...")
check = pd.read_csv(OUT, dtype=str)

assert list(check.columns) == ["query_id", "answer"]
print("[OK] Колонки")

expected_qids = set(benchmark_queries["query_id"].astype(str))
assert set(check["query_id"]) == expected_qids
print(f"[OK] Все {len(expected_qids)} query_id")

assert check["query_id"].nunique() == len(check)
print("[OK] Нет дублей query_id")

check["items"] = check["answer"].str.split(" ")
check["n"]     = check["items"].apply(lambda x: len([i for i in x if i]) if isinstance(x, list) else 0)

assert (check["n"] <= N_CANDS).all()
print(f"[OK] <= {N_CANDS} item_id в строке")

dupes = check["items"].apply(lambda x: len([i for i in x if i]) != len({i for i in x if i}) if isinstance(x, list) else False)
assert not dupes.any()
print("[OK] Нет дублей item_id внутри строки")

all_pred_ids = {i for lst in check["items"] if isinstance(lst, list) for i in lst if i}
invalid = all_pred_ids - VALID_IDS
print(f"[OK] Все item_id из benchmark_items" if not invalid else f"[WARN] {len(invalid)} невалидных ID")

hex_re = re.compile(r"^[0-9a-f]{16}$")
bad    = [i for i in all_pred_ids if not hex_re.match(str(i))]
print("[OK] Формат item_id (16 hex)" if not bad else f"[WARN] {len(bad)} не в формате")

print(f"\n=== ИТОГ ===")
print(f"  Recall@{N_CANDS} [OOD]: {val_recall:.4f}")
print(f"  Строк в answer.csv: {len(check)}")
print(f"  Ср. кандидатов:     {check['n'].mean():.2f}")
print(f"\nанswer.csv готов к отправке!")


Проверяем формат...
[OK] Колонки
[OK] Все 2452 query_id
[OK] Нет дублей query_id
[OK] <= 50 item_id в строке
[OK] Нет дублей item_id внутри строки
[OK] Все item_id из benchmark_items
[OK] Формат item_id (16 hex)

=== ИТОГ ===
  Recall@50 [OOD]: 0.4040
  Строк в answer.csv: 2452
  Ср. кандидатов:     50.00

анswer.csv готов к отправке!
